In [ ]:
from utils import *
import pickle

root_directory = "/nfs/datz/olmo_models/new_outputs/"

In [ ]:
with open(root_directory+"main/books_written.pkl", "rb") as file:
    test = pickle.load(file)

In [ ]:
def graph_signature(G):
    """
    Create a signature for a graph by tagging nodes and edges.
    Returns a set containing tuples for nodes and edges.
    """
    nodes = {("node", n) for n in G.nodes()}
    edges = {("edge", u, v) for u, v in G.edges()}
    return nodes.union(edges)

def jaccard_similarity(G1, G2):
    """
    Compute the Jaccard similarity between two graphs based on their signatures.
    """
    sig1 = graph_signature(G1)
    sig2 = graph_signature(G2)
    intersection = sig1.intersection(sig2)
    union = sig1.union(sig2)
    return len(intersection) / len(union) if union else 1.0

def snapshot_sort_key(snapshot_name):
    """Sort snapshots by step number; 'main' is considered the final snapshot."""
    if snapshot_name == "main":
        return float('inf')
    try:
        # Extract the numeric part after 'step' before the first dash.
        step_str = snapshot_name.split("step")[1].split('-')[0]
        return int(step_str)
    except Exception:
        return 0
    
import networkx as nx

def merge_subgraphs(subgraphs):
    """
    Merge one or more subgraphs into a single graph.
    For duplicate edges, the 'weight' is set as the maximum encountered.
    """
    combined_graph = nx.DiGraph()
    for sg in subgraphs:
        for u, v, data in sg.edges(data=True):
            current_weight = data.get("weight", 0)
            if combined_graph.has_edge(u, v):
                existing_weight = combined_graph[u][v].get("weight", 0)
                combined_graph[u][v]["weight"] = max(existing_weight, current_weight)
            else:
                combined_graph.add_edge(u, v, **data)
    return combined_graph

def collapse_node(node, subj_span, rel_span, ans_span):
    """
    Collapse a node label by removing the token number when it falls into a specified span.
    
    Expected input node name formats:
      - Original: "A0_0" (prefix and token)
      - Already partially renamed: "A0_Subject_0"
    
    For nodes whose token falls in one of the defined spans:
      - Subject: tokens in [subj_span[0], subj_span[1]]
      - Relation: tokens in [rel_span[0], rel_span[1]]
      - Answer: tokens in [ans_span[0], ans_span[1]]
    
    In these cases the output label will be, e.g., "A0_Subject" (without a token number).
    Nodes not falling into any span are left unchanged.
    """
    parts = node.split('_')
    
    # If already in the collapsed form (three parts: e.g. "A0", "Subject", "0")
    if len(parts) == 3:
        # We assume the node is already tagged. Return the collapsed label.
        prefix, role, token_str = parts
        # Even if token_str is there, we ignore it and collapse.
        return f"{prefix}_{role}"
    # If the node is in original two-part format: e.g., "A0_0"
    elif len(parts) == 2:
        prefix, token_str = parts
        try:
            token = int(token_str)
        except Exception:
            return node
        if subj_span[0] <= token <= subj_span[-1]:
            return f"{prefix}_{subj_span[0]}-{subj_span[-1]}"
        elif rel_span[0] <= token <= rel_span[-1]:
            return f"{prefix}_{rel_span[0]}-{rel_span[-1]}"
        elif ans_span[0] <= token <= ans_span[-1]:
            return f"{prefix}_{ans_span[0]}-{ans_span[-1]}"
        else:
            # For nodes that do not fall into any of our special spans, keep original.
            return node
    else:
        return node

def collapse_graph_nodes(G, subj_span, rel_span, ans_span):
    """
    Create a new graph where nodes are collapsed according to the specified spans.
    For each edge in G, we determine the new labels for u and v.
    If an edge already exists in the new graph, we update its weight to be the maximum.
    """
    newG = nx.DiGraph()
    for u, v, data in G.edges(data=True):
        new_u = collapse_node(u, subj_span, rel_span, ans_span)
        new_v = collapse_node(v, subj_span, rel_span, ans_span)
        # If an edge from new_u to new_v already exists, update the weight.
        if newG.has_edge(new_u, new_v):
            newG[new_u][new_v]['weight'] = max(newG[new_u][new_v]['weight'], data.get('weight', 0))
        else:
            newG.add_edge(new_u, new_v, **data)
    return newG

def merge_and_collapse_subgraph(subgraph, subj_span, rel_span, ans_span):
    """
    Process a subgraph (or list of subgraphs) by merging duplicate edges and collapsing
    nodes that fall into the subject, relation, or answer spans.
    
    Parameters:
      subgraph: a NetworkX DiGraph or a list of DiGraph objects.
      subj_span: tuple (min, max) for subject token indices (inclusive).
      rel_span: tuple (min, max) for relation token indices (inclusive).
      ans_span: tuple (min, max) for answer token indices (inclusive).
    
    Returns:
      A new DiGraph with collapsed node labels and merged edge weights.
    """
    # If a single DiGraph is passed, wrap it in a list.
    if isinstance(subgraph, nx.DiGraph):
        subgraph = [subgraph]
    
    merged_graph = merge_subgraphs(subgraph)
    collapsed_graph = collapse_graph_nodes(merged_graph, subj_span, rel_span, ans_span)
    return collapsed_graph

In [ ]:
output_file = os.path.join('./', "all_snapshots_reduced.pkl")
with open(output_file, "rb") as f:
    all_snapshots_reduced = pickle.load(f)

In [ ]:
all_snapshots_reduced['main']

### Measure the effect of the threshhold

In [ ]:
import networkx as nx

def get_output_node(n_layers, token: int):
    return f"I{n_layers - 1}_{token}"

def change_threshold_graph_for_token(graph, threshold, answer_token_pos):
    rgraph = graph.reverse()
    search_graph = nx.subgraph_view(
        rgraph, filter_edge=lambda u, v: rgraph[u][v]["weight"] > threshold
    )
    # Get the node corresponding to the token in the last layer.
    token_node = get_output_node(32, answer_token_pos)
    edges = nx.edge_dfs(search_graph, source=token_node)
    tree = search_graph.edge_subgraph(edges)
    # Reverse the edges because the DFS was performed from the output layer downwards.
    return tree.reverse()

In [ ]:
relation = "official_language"
token_subgraphs1 = all_snapshots_reduced['main'][relation][0]["token_subgraphs"][-2]
token_subgraphs2 = all_snapshots_reduced['main'][relation][1]["token_subgraphs"][-2]

answer_token_pos1 = all_snapshots_reduced['main'][relation][0]["answer_token_span"][0] - 1
answer_token_pos2 = all_snapshots_reduced['main'][relation][1]["answer_token_span"][0] - 1

print(f"Size of new_graph1 {token_subgraphs1.number_of_nodes()}, {token_subgraphs1.number_of_edges()}")
print(f"Size of new_graph1 {token_subgraphs2.number_of_nodes()}, {token_subgraphs2.number_of_edges()}")

new_graph1 = change_threshold_graph_for_token(token_subgraphs1, 0.03, answer_token_pos1)
new_graph2 = change_threshold_graph_for_token(token_subgraphs2, 0.03, answer_token_pos2)
print(f"Size of new_graph1 {new_graph1.number_of_nodes()}, {new_graph1.number_of_edges()}")
print(f"Size of new_graph1 {new_graph2.number_of_nodes()}, {new_graph2.number_of_edges()}")

We reduced the relations to examples which have a single token answer.

In [ ]:
all_snapshots_reduced['main'][relation][0]['subj_token_span']

### Comparison between Graph and Collapsed Graph

In [ ]:
relation = "country_capital_city"
snapshot = "main"#"step5000-tokens20B"

subj_token_span0 = all_snapshots_reduced[snapshot][relation][0]['subj_token_span']
answer_token_span0 = all_snapshots_reduced[snapshot][relation][0]['answer_token_span']

subj_token_span1 = all_snapshots_reduced[snapshot][relation][1]['subj_token_span']
answer_token_span1 = all_snapshots_reduced[snapshot][relation][1]['answer_token_span']

subject_span0 = subj_token_span0
relation_span0 = [subj_token_span0[-1]+1, answer_token_span0[0]-2] 
answer_span0  = [x -1 for x in answer_token_span0]

subject_span1 = subj_token_span1
relation_span1 = [subj_token_span1[-1]+1, answer_token_span1[0]-2] 
answer_span1  = [x - 1 for x in answer_token_span1]

In [ ]:
token_subgraphs0 = all_snapshots_reduced[snapshot][relation][0]["token_subgraphs"][-2]
token_subgraphs1 = all_snapshots_reduced[snapshot][relation][1]["token_subgraphs"][-2]

print(f"Example: {all_snapshots_reduced[snapshot][relation][0]['sentence']}, with {len(all_snapshots_reduced['main'][relation][0]['token_subgraphs'])} tokens")
print(f"Nodes: {len(token_subgraphs0.nodes())}, Edges: {len(token_subgraphs0.edges())}")

print(f"Example: {all_snapshots_reduced[snapshot][relation][1]['sentence']}, with {len(all_snapshots_reduced['main'][relation][1]['token_subgraphs'])} tokens")
print(f"Nodes: {len(token_subgraphs1.nodes())}, Edges: {len(token_subgraphs1.edges())}")


print(f"Jaccard Similarity {jaccard_similarity(token_subgraphs0, token_subgraphs1)}")


# new_graph1 = change_threshold_graph_for_token(token_subgraphs0, 0.03, answer_token_span0[0]-1)
# new_graph2 = change_threshold_graph_for_token(token_subgraphs1, 0.03, answer_token_span1[0]-1)
# print(f"Size of new_graph1 {new_graph1.number_of_nodes()}, {new_graph1.number_of_edges()}")
# print(f"Size of new_graph1 {new_graph2.number_of_nodes()}, {new_graph2.number_of_edges()}")

In [ ]:
token_subgraphs0 = all_snapshots_reduced[snapshot][relation][0]["token_subgraphs"][-2]
token_subgraphs1 = all_snapshots_reduced[snapshot][relation][1]["token_subgraphs"][-2]

result_graph0 = merge_and_collapse_subgraph(token_subgraphs0, subject_span0, relation_span0, answer_span0)
result_graph1 = merge_and_collapse_subgraph(token_subgraphs1, subject_span1, relation_span1, answer_span1)

print(f"Example: {all_snapshots_reduced[snapshot][relation][0]['sentence']}")
print(f"Nodes: {len(result_graph0.nodes())}, Edges: {len(result_graph0.edges())}")

print(f"Example: {all_snapshots_reduced[snapshot][relation][1]['sentence']}")
print(f"Nodes: {len(result_graph1.nodes())}, Edges: {len(result_graph1.edges())}")

print(f"Jaccard Similarity {jaccard_similarity(result_graph0, result_graph1)}")

In [ ]:
result_graph0.nodes()

In [ ]:
import networkx as nx

def merge_subgraphs(subgraphs):
    # Create an empty directed graph for the combined result.
    combined_graph = nx.DiGraph()
    
    # Iterate over each subgraph in the list.
    for sg in subgraphs:
        for u, v, data in sg.edges(data=True):
            # Extract the score for the current edge; adjust the default value as needed.
            current_score = data.get("score", 0)
            
            # If the combined graph already has the edge, update the score with the maximum value.
            if combined_graph.has_edge(u, v):
                existing_score = combined_graph[u][v].get("score", 0)
                combined_graph[u][v]["score"] = max(existing_score, current_score)
            else:
                # Otherwise, add the edge with its attributes.
                combined_graph.add_edge(u, v, **data)
    
    return combined_graph

In [ ]:
test = merge_subgraphs(token_subgraphs)

In [ ]:
# Specify the relation to analyze.
#relation_to_analyze = "city_in_country"  # Adjust if needed.

# Load your all_snapshots_reduced dictionary.
# For example:
# with open("all_snapshots_reduced.pkl", "rb") as f:
#     all_snapshots_reduced = pickle.load(f)

# Sort the snapshot names.
sorted_snapshots = sorted(all_snapshots_reduced.keys(), key=snapshot_sort_key)

# Lists to store snapshots and the corresponding Jaccard similarity.
snapshots_list = []
jaccard_similarities = []

for snapshot in sorted_snapshots:
    relations = all_snapshots_reduced[snapshot]
    if relation in relations:
        entries = relations[relation]
        if len(entries) >= 2:
            entry0 = entries[0]
            entry1 = entries[1]
            token_subgraphs0 = entry0.get('token_subgraphs', [])
            token_subgraphs1 = entry1.get('token_subgraphs', [])
            if len(token_subgraphs0) >= 2 and len(token_subgraphs1) >= 2:
                # Use the second-to-last token subgraph.
                graph0 = token_subgraphs0[entry0.get('answer_token_span')[0]-1]
                graph1 = token_subgraphs1[entry1.get('answer_token_span')[0]-1]
                
                #new_graph0 = change_threshold_graph_for_token(graph0, 0.03, answer_token_pos1)
                #new_graph1 = change_threshold_graph_for_token(graph1, 0.03, answer_token_pos2)
                
                try:
                    sim = jaccard_similarity(graph0, graph1)
                    snapshots_list.append(snapshot)
                    jaccard_similarities.append(sim)
                except Exception as err:
                    print(f"Error processing snapshot '{snapshot}': {err}")
            else:
                print(f"Snapshot '{snapshot}' does not have at least 2 token_subgraphs for both entries.")
        else:
            print(f"Snapshot '{snapshot}' does not have 2 entries for relation '{relation}'.")
    else:
        print(f"Relation '{relation}' not found in snapshot '{snapshot}'.")

# Plot the Jaccard similarities over snapshots as one line graph.
plt.figure(figsize=(10, 6))
plt.plot(snapshots_list, jaccard_similarities, marker='o', linestyle='-')
plt.xlabel("Snapshot")
plt.ylim(0,1)
plt.ylabel("Jaccard Similarity")
plt.title(f"Jaccard Similarity of token_subgraphs for '{relation}'")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
snapshots_list = []
jaccard_similarities = []

for snapshot in sorted_snapshots:
    relations = all_snapshots_reduced[snapshot]
    if relation in relations:
        entries = relations[relation]
        if len(entries) >= 2:
            entry0 = entries[0]
            entry1 = entries[1]
            token_subgraphs0 = entry0.get('token_subgraphs', [])
            token_subgraphs1 = entry1.get('token_subgraphs', [])
            if len(token_subgraphs0) >= 2 and len(token_subgraphs1) >= 2:
                # Use the second-to-last token subgraph.
                graph0 = token_subgraphs0[entry0.get('answer_token_span')[0]-1]
                graph1 = token_subgraphs1[entry1.get('answer_token_span')[0]-1]
                
                #new_graph0 = change_threshold_graph_for_token(graph0, 0.03, answer_token_pos1)
                #new_graph1 = change_threshold_graph_for_token(graph1, 0.03, answer_token_pos2)
                
                result_graph0 = merge_and_collapse_subgraph(graph0, subject_span0, relation_span0, answer_span0)
                result_graph1 = merge_and_collapse_subgraph(graph1, subject_span1, relation_span1, answer_span1)

                try:
                    sim = jaccard_similarity(result_graph0, result_graph1)
                    snapshots_list.append(snapshot)
                    jaccard_similarities.append(sim)
                except Exception as err:
                    print(f"Error processing snapshot '{snapshot}': {err}")
            else:
                print(f"Snapshot '{snapshot}' does not have at least 2 token_subgraphs for both entries.")
        else:
            print(f"Snapshot '{snapshot}' does not have 2 entries for relation '{relation}'.")
    else:
        print(f"Relation '{relation}' not found in snapshot '{snapshot}'.")

# Plot the Jaccard similarities over snapshots as one line graph.
plt.figure(figsize=(10, 6))
plt.plot(snapshots_list, jaccard_similarities, marker='o', linestyle='-')
plt.xlabel("Snapshot")
plt.ylim(0,1)
plt.ylabel("Jaccard Similarity")
plt.title(f"Jaccard Similarity of token_subgraphs for '{relation}'")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
test = all_snapshots_reduced[snapshot][relation][0]["token_subgraphs"][-2]

In [ ]:
c_attns = all_snapshots_reduced[snapshot][relation][0]['contributions']['c_attns']

In [ ]:
t1 = get_head_contribution_map(test, c_attns)

In [ ]:
t1.sum()

In [ ]:
def get_head_contribution_map(gr: nx.DiGraph, attn_contr_maps, THR=0.0):
    """
    For each edge in gr, parses s and t which may look like "A24_1-4" or "A0_5-5",
    extracts all integer token positions in each span, fetches the
    attn_contr_maps[layer][0] block (shape [seq, seq, heads]), and then
    sees which heads exceed THR *for any* (target, source) pair in the block.
    
    Returns:
      contr_map: bool array of shape [num_layers, num_heads]
    """
    num_layers = len(attn_contr_maps)
    num_heads  = attn_contr_maps[0].shape[-1]
    contr_map  = np.zeros((num_layers, num_heads), dtype=bool)

    def parse_indices(tok_str):
        """ "3" -> [3],   "1-4" -> [1,2,3,4] """
        if "-" in tok_str:
            start, end = map(int, tok_str.split("-"))
            return list(range(start, end+1))
        else:
            return [int(tok_str)]

    for s, t, data in gr.edges(data=True):
        # only look at attention‑type targets
        if not t.startswith("A"):
            continue

        # layer is the number after the leading "A"
        try:
            layer = int(t.split("_")[0][1:])
        except ValueError:
            continue

        # parse the token‐span for source and target
        try:
            t_span = parse_indices(t.split("_",1)[1])
            s_span = parse_indices(s.split("_",1)[1])
        except Exception:
            # if it doesn’t fit the pattern, skip
            continue

        # grab the contribution map for this layer: shape [seq_len, seq_len, heads]
        attn_layer = attn_contr_maps[layer][0].cpu().numpy()

        # extract the little block [len(t_span), len(s_span), heads]
        # np.ix_ makes the outer two axes
        block = attn_layer[np.ix_(t_span, s_span, range(num_heads))]

        # see which heads in that block exceed THR anywhere
        # result is a bool vector of shape [heads]
        head_active = (block > THR).any(axis=(0,1))

        # accumulate into our boolean map
        contr_map[layer] |= head_active

    return contr_map

In [ ]:
col_graph = collapse_graph_nodes(test, subject_span0, relation_span0, answer_span0)

In [ ]:
col_graph.nodes()

In [ ]:
t2 = get_head_contribution_map(col_graph, c_attns)

In [ ]:
t2.sum()

In [ ]:
print(all_snapshots_reduced['main']['country_capital_city'][0]["sentence"])

In [ ]:
print(all_snapshots_reduced['main']['city_in_country'][0]["token_subgraphs"])
print(all_snapshots_reduced['main']['city_in_country'][1]["token_subgraphs"])
print(all_snapshots_reduced['main']['city_in_country'][0]["subj_token_span"])
print(all_snapshots_reduced['main']['city_in_country'][0]["answer_token_span"])
print(all_snapshots_reduced['main']['city_in_country'][1]["subj_token_span"])
print(all_snapshots_reduced['main']['city_in_country'][1]["answer_token_span"])

In [ ]:
subject_span0 = (0, 0)
relation_span0 = (1, 6)
answer_span0  = (7, 7)

# Example usage if you only have one subgraph:
single_subgraph0 = all_snapshots_reduced['main']['country_capital_city'][0]["token_subgraphs"][-2]  # or however you get it
result_graph0 = merge_and_collapse_subgraph(single_subgraph0, subject_span0, relation_span0, answer_span0)

print("Renamed nodes:", list(result_graph0.nodes()))
print("Collapsed (Merged) Edges with Data:")
for edge in result_graph0.edges(data=True):
    print(edge)

In [ ]:
subject_span1 = (0, 1)
relation_span1 = (2, 6)
answer_span1  = (7, 7)

# Example usage if you only have one subgraph:
single_subgraph1 = all_snapshots_reduced['main']['country_capital_city'][1]["token_subgraphs"][-2]  # or however you get it
result_graph1 = merge_and_collapse_subgraph(single_subgraph1, subject_span1, relation_span1, answer_span1)

print("Renamed nodes:", list(result_graph1.nodes()))
print("Collapsed (Merged) Edges with Data:")
for edge in result_graph1.edges(data=True):
    print(edge)

In [ ]:
print(jaccard_similarity(single_subgraph0, single_subgraph1))
print(len(single_subgraph0.nodes()))
print(len(single_subgraph1.nodes()))

print(len(single_subgraph0.edges()))
print(len(single_subgraph1.edges()))



In [ ]:
jaccard_similarity(result_graph0, result_graph1)

print(len(result_graph0.nodes()))
print(len(result_graph1.nodes()))

print(len(result_graph0.edges()))
print(len(result_graph1.edges()))

In [ ]:
sorted_snapshots = sorted(all_snapshots_reduced.keys(), key=snapshot_sort_key)
# Store the similarity for each snapshot
snapshot_similarities = []

subject_span0 = (0, 3)
relation_span0 = (4, 9)
answer_span0  = (10, 10)

subject_span1 = (0, 2)
relation_span1 = (3, 8)
answer_span1  = (9, 9)

# Iterate over each snapshot
for idx, snapshot in enumerate(sorted_snapshots):
    # Select two subgraphs from the snapshot.
    # Adjust the indices as needed (here we pick the second-to-last for graph1 and the last for graph2).
    subgraph0 = all_snapshots_reduced[snapshot]['city_in_country'][0]["token_subgraphs"][-2]
    subgraph1 = all_snapshots_reduced[snapshot]['city_in_country'][1]["token_subgraphs"][-2]
    
    # Merge and collapse each subgraph with its corresponding span parameters.
    collapsed_graph1 = merge_and_collapse_subgraph(subgraph0, subject_span0, relation_span0, answer_span0)
    collapsed_graph2 = merge_and_collapse_subgraph(subgraph1, subject_span1, relation_span1, answer_span1)
    
    # Compute the Jaccard similarity between the resulting collapsed graphs.
    sim = jaccard_similarity(collapsed_graph1, collapsed_graph2)
    snapshot_similarities.append(sim)
    
    # Optionally, print the details for each snapshot.
    print(f"Snapshot {snapshot}:")
    print("  Graph 1 Nodes:", list(collapsed_graph1.nodes()))
    print("  Graph 1 Edges:")
    for edge in collapsed_graph1.edges(data=True):
        print("    ", edge)
    print("  Graph 2 Nodes:", list(collapsed_graph2.nodes()))
    print("  Graph 2 Edges:")
    for edge in collapsed_graph2.edges(data=True):
        print("    ", edge)
    print(f"  Jaccard Similarity: {sim:.3f}\n")

# ----------------------
# Plot the Similarity Over Snapshots
# ----------------------
plt.figure(figsize=(8, 5))
plt.plot(range(len(snapshot_similarities)), snapshot_similarities, marker='o')
plt.xlabel("Snapshot Index")
plt.ylabel("Jaccard Similarity")
plt.title("Jaccard Similarity Between Two Subgraphs Over Snapshots")
plt.grid(True)
plt.show()